In [ ]:
!pip install ultralytics 'sng4onnx>=1.0.1' 'onnx_graphsurgeon>=0.3.26' 'ai-edge-litert>=1.2.0' 'onnx>=1.12.0,<2.0.0' 'onnx2tf>=1.26.3,<1.29.0' 'onnxslim>=0.1.71' 'onnxruntime' 'onnxruntime-gpu'

In [ ]:
!pip install -U ultralytics wandb

input link ke dataset

In [ ]:
!mkdir "/content/dataset"

In [ ]:
!yolo settings wandb=True

In [ ]:
from ultralytics import YOLO
import wandb as wb
import os
from google.colab import userdata


Di tab kanan ada secrets, centang notebook access

In [ ]:
def convert_model(model, fraction=0.06, imgsz=640):
  tflite = model.export(format="tflite", data=f"{dataset_dir}/data.yaml", device="cpu", int8=True, fraction=fraction, imgsz=imgsz)
  return tflite

In [ ]:
class WandbHandler:
  def __init__(self, project_name, root='/content/'):
    self.project_name = project_name
    self.root = root
    wb.login(key=userdata.get('WANDB_API_KEY'))

  def download_from_wandb(self, dataset_name):
    dataset_root = f"{self.root}dataset/{dataset_name}"
    run = wb.init(project=self.project_name, job_type="training")
    artifact = run.use_artifact(f"{self.project_name}/{dataset_name}:latest")

    if os.path.exists(dataset_root) and len(os.listdir(dataset_root)) > 0:
      print(f"Dataset already exists and is populated at {dataset_root}. Skipping download.")
      return run, dataset_root

    artifact.download(root=dataset_root)
    return run, dataset_root

  def upload_to_wandb(self, dataset_dir):
    artifact = wb.Artifact(name=dataset_dir, type="dataset")
    artifact.add_dir(self.root + dataset_dir)
    with wb.init(project=self.project_name, job_type='upload_dataset') as run:
      run.log_artifact(artifact)

  def update_dataset(self, dataset_dir):
    with wb.init(project=self.project_name, job_type='update_dataset') as run:
      artifact = wb.Artifact(name=dataset_dir, type="dataset")
      artifact.add_dir(self.root + dataset_dir)
      run.log_artifact(artifact)


  def update_file(self, dataset_dir, file_name):
    with wb.init(project=self.project_name, job_type='update_file') as run:
      saved_artifact = run.use_artifact(dataset_dir+":latest")
      draft_artifact = saved_artifact.new_draft()

      draft_artifact.remove(saved_artifact.get_entry(file_name))
      draft_artifact.add_file(local_path= self.root + dataset_dir + "/" + file_name, name=file_name)

      draft_artifact.save()


  def resume_run(self, run_id, dataset_name):
    dataset_root = f"{self.root}dataset/{dataset_name}"
    run = wb.init(project=self.project_name, job_type="training", id=run_id, resume="allow")
    dataset = run.use_artifact(f"{self.project_name}/{dataset_name}:latest")
    checkpoint_root = f"{self.root}{run_id}_checkpoint"
    checkpoint = run.use_artifact(f"{self.project_name}/{run_id}_checkpoint:latest")


    if os.path.exists(checkpoint_root) and len(os.listdir(checkpoint_root)) > 0:
      print(f"Checkpoint already exists and is populated at {checkpoint_root}. Skipping download.")
    else:
      checkpoint.download(root=checkpoint_root)

    if os.path.exists(dataset_root) and len(os.listdir(dataset_root)) > 0:
      print(f"Dataset already exists and is populated at {dataset_root}. Skipping download.")
    else:
      dataset.download(root=dataset_root)

    return run, checkpoint_root, dataset_root


  def get_model_for_conversion(self, model_name, dataset_name):
    api = wb.Api()
    dataset_root = f"{self.root}dataset/{dataset_name}"
    model_root = f"{self.root}{model_name}"
    dataset = api.artifact(f"{self.project_name}/{dataset_name}:latest")
    model = api.artifact(f"{self.project_name}/{model_name}:latest")

    if os.path.exists(model_root) and len(os.listdir(model_root)) > 0:
      print(f"Model already exists and is populated at {model_root}. Skipping download.")
    else:
      model.download(root=model_root)

    if os.path.exists(dataset_root) and len(os.listdir(dataset_root)) > 0:
      print(f"Dataset already exists and is populated at {dataset_root}. Skipping download.")
    else:
      dataset.download(root=dataset_root)

    return model_root, dataset_root

  def update_mobile_model(self, mobile_dir, model_name, imgsz=640):
    with wb.init(project=self.project_name, job_type='update_mobile') as run:
        mobile_name = model_name.split('_')
        mobile_name[-1] = "mobile"
        model_type = mobile_name[0]
        mobile_name = '_'.join(mobile_name)
        saved_artifact = run.use_artifact(f"{self.project_name}/{mobile_name}:latest")
        draft_artifact = saved_artifact.new_draft()
        draft_artifact.add_file(local_path=mobile_dir, name=f"{model_type}_{imgsz}_int8.tflite")
        run.log_artifact(draft_artifact, aliases=["latest", "best"])




# Parameter Konversi
Setting parameter konversi disini. Pastikan dataset yg digunakan utk konversi sama dengan dataset train.

In [ ]:


project = "FAIt (Food AI-based tracking)"
dataset_name = "Food_Computer_Vision_Dataset" # nama dataset yang sudah ada di Weights & Biases yang digunakan utk train
model_name = "yolo26n_Food_Computer_Vision_Dataset_100_2026-03-27-15-13_model" # model yang akan dikonversi
wandb_handler = WandbHandler(project)
imgsz=320 # ukuran input gambar, kecilkan dari train agar model lebih cepat
fraction=0.1 #persentase jumlah gambar di set valid yang digunakan, min 300-500 utk dataset besar, min 50 utk dataset
model_dir, dataset_dir = wandb_handler.get_model_for_conversion(model_name, dataset_name)

In [ ]:
model = YOLO(f"{model_dir}/best.pt")
tflite = convert_model(model, fraction=fraction, imgsz=imgsz)

In [ ]:
wandb_handler = WandbHandler(project)
wandb_handler.update_mobile_model(tflite, model_name, imgsz=imgsz)